# Train GNN Model

## Overall Workflow
1. Read packages from unknown_packages.txt(unknown group) and vulnerable_packages.txt(positive group) and combine them into one collection
2. Initialize a FeatureGenerator with system = "pypi"
3. Initialize a GNN model with two layers of propagation(corresponding to the default bfs-depth in GraphGenerator)
4. Iterate through package + version:
    + Instantiate a GraphGenerator instance with the package + version
    + Generate full features for the package + version using FeatureGenerator's function of get_full_features
    + Append the package's embeddings into the tensor/dataframe initialized earlier, also save the graph information together
5. Split resulting features + graph info collections into training set(for training GNN) and testing set(for testing clustering performance)
6. Train the GNN model(GraphSAGE) with GCL (Graph Contrastive Learning) + training set.
7. Using testing sets, input the features and its corresponding graph structure into the GNN and get the GNN processed package embeddings(for each package)
8. Fit a clustering Model (K-means) on these package embeddings (Tune K use Elbow Method)
9. Evaluate the clustering result (best K)
10. Save the GNN model

## Setup

In [1]:
import sys
import os
import pickle
import random

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

from scripts.FeatureGenerator import FeatureGenerator
from scripts.GraphGenerator import GraphGenerator, configure_dependency_cache
from models.gnn_encoder import GCLModel
from models.graph_utils import build_pyg_data
from models.gcl_trainer import GCLTrainer

# ── Paths ──────────────────────────────────────────────────────────────────
VULNERABLE_FILE = '../data/vulnerable_packages.txt'
UNKNOWN_FILE    = '../data/unknown_packages.txt'
METADATA_CACHE  = '../data/package_metadata_cache.csv'            # shared with baseline
DEPS_CACHE      = '../data/depsdev_deps_cache.json'
GRAPH_CACHE     = '../data/gnn_graph_metadata_only_dataset.pkl'   # metadata-only PyG graphs
MODEL_OUTPUT    = '../models/lib/gnn_encoder2.pt'
KMEANS_OUTPUT   = '../models/lib/gnn_kmeans2.pkl'
SCALER_OUTPUT   = '../models/lib/gnn_scaler2.pkl'

# ── Hyper-parameters ───────────────────────────────────────────────────────
FEATURE_DIM     = 20   # metadata-only: structural features dropped (full = 27)
HIDDEN_CHANNELS = 64
EMBED_DIM       = 32
PROJ_HIDDEN_DIM = 64
PROJ_OUT_DIM    = 32
NUM_LAYERS      = 2    # matches BFS depth of GraphGenerator
NUM_EPOCHS      = 50
BATCH_SIZE      = 32
LR              = 1e-3
TEMPERATURE     = 0.5
TRAIN_RATIO     = 0.8
TOP_K_PCT       = 0.33

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs('../models/lib', exist_ok=True)
os.makedirs('../data/cache', exist_ok=True)

print(f'Device : {DEVICE}')
print(f'Feature dim : {FEATURE_DIM}')
print('Setup complete.')

Device : cpu
Feature dim : 20
Setup complete.


## Step 1–2: Load packages and initialise FeatureGenerator

In [2]:
def load_packages(filepath, label):
    entries = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                pkg, ver = line.rsplit('@', 1)
                entries.append({'package': pkg, 'version': ver, 'label': label})
    return entries

vulnerable_entries = load_packages(VULNERABLE_FILE, label=1) # For test, use first 500
unknown_entries    = load_packages(UNKNOWN_FILE,    label=0) # For test, use first 500
all_entries        = vulnerable_entries + unknown_entries

# Initialise FeatureGenerator (uses shared metadata cache with baseline model)
fg = FeatureGenerator(system='pypi', cache_path=METADATA_CACHE)

# Load persistent dependency cache (shared with dataset construction scripts)
configure_dependency_cache(DEPS_CACHE)

print(f'Vulnerable : {len(vulnerable_entries)}')
print(f'Unknown    : {len(unknown_entries)}')
print(f'Total      : {len(all_entries)}')

[FeatureGenerator] Loaded 32790 cached entries from ../data/package_metadata_cache.csv
[GraphGenerator] Loaded 22060 cached dependency entries from ../data/depsdev_deps_cache.json
Vulnerable : 5000
Unknown    : 5000
Total      : 10000


## Step 3–4: Build graph dataset

For each package:
1. Build a dependency graph via `GraphGenerator`
2. Extract full features (package + structural) for every node via `FeatureGenerator.get_full_features`
3. Convert to a PyG `Data` object with `build_pyg_data`

Results are cached to `gnn_graph_dataset.pkl`. **Skip to the *Load cache* cell if the cache exists.**

In [3]:
# import time

# graph_dataset = []   # list of dicts: {data, label, pkg_id}
# failed        = []
# pkg_times     = []   # per-package elapsed seconds
# total         = len(all_entries)

# pbar = tqdm(all_entries, desc='Building graphs')
# for entry in pbar:
#     pkg    = entry['package']
#     ver    = entry['version']
#     label  = entry['label']
#     pkg_id = f'{pkg}@{ver}'

#     t_start = time.perf_counter()
#     try:
#         gg        = GraphGenerator(pkg, ver)
#         nodes_map = gg.nodes_map

#         # Extract full (package + structural) features for every node in the subgraph
#         features_dict = {}
#         for node_id in nodes_map:
#             try:
#                 result = fg.get_full_features(node_id, nodes_map)
#                 features_dict[node_id] = result['full_metadata']
#             except Exception:
#                 features_dict[node_id] = [0.0] * FEATURE_DIM

#         data, _ = build_pyg_data(nodes_map, features_dict, root_id=pkg_id)

#         if data.x.size(0) == 0:
#             failed.append(pkg_id)
#         else:
#             graph_dataset.append({'data': data, 'label': label, 'pkg_id': pkg_id})

#     except Exception as e:
#         failed.append(pkg_id)

#     elapsed = time.perf_counter() - t_start
#     pkg_times.append(elapsed)

#     # Real-time stats in the progress bar suffix
#     n_done      = len(graph_dataset)
#     n_failed    = len(failed)
#     n_processed = n_done + n_failed
#     pct_ok      = n_done   / n_processed * 100 if n_processed else 0.0
#     pct_fail    = n_failed / n_processed * 100 if n_processed else 0.0
#     avg_t       = sum(pkg_times) / len(pkg_times)
#     pbar.set_postfix({
#         'ok'  : f'{n_done}/{n_processed} ({pct_ok:.1f}%)',
#         'fail': f'{n_failed} ({pct_fail:.1f}%)',
#         't(s)': f'{elapsed:.2f} | avg {avg_t:.2f}',
#     })

# # Persist to cache
# with open(GRAPH_CACHE, 'wb') as f:
#     pickle.dump(graph_dataset, f)

# # Final timing summary
# print(f'\nCollected : {len(graph_dataset)} / {total} ({len(graph_dataset)/total*100:.1f}%)')
# print(f'Failed    : {len(failed)} / {total} ({len(failed)/total*100:.1f}%)')
# print(f'Time/pkg  : min={min(pkg_times):.2f}s  max={max(pkg_times):.2f}s  avg={sum(pkg_times)/len(pkg_times):.2f}s  total={sum(pkg_times):.1f}s')

In [4]:
# Load cache (run this instead of the build cell if cache already exists)
with open(GRAPH_CACHE, 'rb') as f:
    graph_dataset = pickle.load(f)

# Filter out any cached graphs with empty node feature matrices
before = len(graph_dataset)
graph_dataset = [d for d in graph_dataset if d['data'].x.size(0) > 0]
n_empty = before - len(graph_dataset)

n_vuln = sum(1 for d in graph_dataset if d['label'] == 1)
print(f'Loaded {len(graph_dataset)} graphs  ({n_vuln} vulnerable)'
      + (f'  [{n_empty} empty graphs dropped]' if n_empty else ''))

Loaded 9668 graphs  (4823 vulnerable)


## Step 5: Train / test split

In [5]:
random.seed(42)
shuffled = graph_dataset.copy()
random.shuffle(shuffled)

split_idx   = int(len(shuffled) * TRAIN_RATIO)
train_set   = shuffled[:split_idx]
test_set    = shuffled[split_idx:]

# Unpack into (Data, root_idx) pairs expected by GCLTrainer
train_pairs = [(item['data'], 0) for item in train_set]
test_pairs  = [(item['data'], 0) for item in test_set]
test_labels = [item['label']     for item in test_set]
test_pkgids = [item['pkg_id']    for item in test_set]

print(f'Train : {len(train_pairs)}')
print(f'Test  : {len(test_pairs)}')

Train : 7734
Test  : 1934


## Step 6: GCL Training

For each batch of graphs:
1. Create **G₁** and **G₂** via independent random augmentations (edge dropout / feature masking / leaf removal / feature noise)
2. Forward both views through the shared GraphSAGE encoder
3. Extract the root-node embedding `h_root`, pass through the projection head, and L2-normalise
4. Compute **InfoNCE** loss — maximise `sim(p₁, p₂)` for the same graph, minimise it against all other graphs in the batch
5. Back-propagate and update encoder + projection head

After training the projection head is discarded; only the encoder weights are retained.

In [6]:
# ── Initialise model (Step 3 of the overall workflow) ─────────────────────
model = GCLModel(
    in_channels     = FEATURE_DIM,
    hidden_channels = HIDDEN_CHANNELS,
    embed_dim       = EMBED_DIM,
    proj_hidden_dim = PROJ_HIDDEN_DIM,
    proj_out_dim    = PROJ_OUT_DIM,
    num_layers      = NUM_LAYERS,
)
trainer = GCLTrainer(model, lr=LR, temperature=TEMPERATURE, device=DEVICE)

print(model)
print(f'\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

GCLModel(
  (encoder): GraphSAGEEncoder(
    (convs): ModuleList(
      (0): SAGEConv(20, 64, aggr=mean)
      (1): SAGEConv(64, 32, aggr=mean)
    )
    (bns): ModuleList(
      (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
  )
  (projection_head): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=64, out_features=32, bias=True)
    )
  )
)

Trainable parameters: 11,200


In [7]:
# ── Training loop ─────────────────────────────────────────────────────────
epoch_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    # Shuffle training graphs each epoch
    random.shuffle(train_pairs)

    # Mini-batch loop
    batch_losses = []
    for start in range(0, len(train_pairs), BATCH_SIZE):
        batch = train_pairs[start : start + BATCH_SIZE]
        if len(batch) < 2:
            continue   # InfoNCE needs at least 2 samples
        loss = trainer.train_epoch(batch)
        batch_losses.append(loss)

    avg_loss = float(np.mean(batch_losses)) if batch_losses else float('nan')
    epoch_losses.append(avg_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  loss={avg_loss:.4f}')

print('\nTraining complete.')
print('Projection head will be discarded — only encoder weights are kept for inference.')

Epoch   1/50  loss=3.0526
Epoch  10/50  loss=2.7857
Epoch  20/50  loss=2.7216
Epoch  30/50  loss=2.6914
Epoch  40/50  loss=2.6665
Epoch  50/50  loss=2.6507

Training complete.
Projection head will be discarded — only encoder weights are kept for inference.


## Step 7: Get test-set embeddings

Pass each test graph through the **encoder only** (no projection head) to obtain the root-node package embedding.

In [8]:
embeddings = []

for data, root_idx in tqdm(test_pairs, desc='Extracting embeddings'):
    emb = trainer.get_embedding(data, root_idx=root_idx)   # (embed_dim,)
    embeddings.append(emb.numpy())

X_embed = np.array(embeddings)   # (N_test, embed_dim)
labels  = np.array(test_labels)

print(f'Embedding matrix : {X_embed.shape}')
print(f'Vulnerable in test : {labels.sum()} / {len(labels)}')

Extracting embeddings: 100%|██████████| 1934/1934 [00:00<00:00, 6902.07it/s]

Embedding matrix : (1934, 32)
Vulnerable in test : 989 / 1934


## Step 8: K-Means clustering — Elbow Method

Sweep K from 2 to 20 and detect the elbow as the K with the maximum second-difference in inertia.

In [9]:
# Standardise embeddings before K-Means (sensitive to scale)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_embed)

K_RANGE  = range(2, 21)
inertias = {}

for k in tqdm(K_RANGE, desc='Elbow sweep'):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias[k] = km.inertia_

elbow_df = pd.DataFrame({
    'K'      : list(inertias.keys()),
    'Inertia': list(inertias.values()),
})
elbow_df['Delta']    = elbow_df['Inertia'].diff().abs()
elbow_df['Delta2']   = elbow_df['Delta'].diff().abs()
elbow_df['Gain_pct'] = (elbow_df['Delta'] / elbow_df['Inertia'].shift(1) * 100).round(2)

print(elbow_df.to_string(index=False))

best_k = int(elbow_df.loc[elbow_df['Delta2'].idxmax(), 'K'])
print(f'\nAuto-detected elbow K = {best_k}')

Elbow sweep: 100%|██████████| 19/19 [00:00<00:00, 21.46it/s]

 K      Inertia        Delta      Delta2  Gain_pct
 2 53797.031250          NaN         NaN       NaN
 3 43385.214844 10411.816406         NaN     19.35
 4 36809.046875  6576.167969 3835.648438     15.16
 5 31000.492188  5808.554688  767.613281     15.78
 6 26770.097656  4230.394531 1578.160156     13.65
 7 23481.791016  3288.306641  942.087891     12.28
 8 21648.402344  1833.388672 1454.917969      7.81
 9 20174.218750  1474.183594  359.205078      6.81
10 18975.404297  1198.814453  275.369141      5.94
11 18082.650391   892.753906  306.060547      4.70
12 17158.609375   924.041016   31.287109      5.11
13 16557.976562   600.632812  323.408203      3.50
14 15758.204102   799.772461  199.139648      4.83
15 15086.525391   671.678711  128.093750      4.26
16 14647.824219   438.701172  232.977539      2.91
17 14088.048828   559.775391  121.074219      3.82
18 13669.807617   418.241211  141.534180      2.97
19 13268.021484   401.786133   16.455078      2.94
20 12924.762695   343.258789   

In [10]:
# Override best_k here if you disagree with the auto-detection
best_k = 4

final_km      = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(X_scaled)

results_df = pd.DataFrame({
    'pkg_id' : test_pkgids,
    'label'  : labels,
    'cluster': cluster_labels,
})

print(f'Trained KMeans with K={best_k}')
print(results_df.groupby('cluster')['label'].agg(['sum', 'count']).rename(
    columns={'sum': 'n_vulnerable', 'count': 'cluster_size'}
))

Trained KMeans with K=4
         n_vulnerable  cluster_size
cluster                            
0                 249           614
1                 487          1053
2                   3             7
3                 250           260


In [11]:
from sklearn.metrics import silhouette_score

sil = silhouette_score(X_scaled, cluster_labels)
print(f'Silhouette score (K={best_k}, GNN embeddings — metadata-only): {sil:.4f}')
print('  > 0.5   → strong structure')
print('  0.2–0.5 → moderate structure')
print('  < 0.2   → weak / overlapping clusters')

Silhouette score (K=4, GNN embeddings — metadata-only): 0.2287
  > 0.5   → strong structure
  0.2–0.5 → moderate structure
  < 0.2   → weak / overlapping clusters


## Step 9: Evaluate

**Risk score** per cluster = `n_vulnerable / cluster_size`  
**Top-K clusters** = top 10 % of clusters ranked by risk score

Metrics:
- **Recall** — fraction of all known-vulnerable packages that fall in the top-K clusters
- **Precision** — fraction of packages inside the top-K clusters that are actually vulnerable

In [12]:
cluster_stats = (
    results_df.groupby('cluster')['label']
    .agg(n_vulnerable='sum', cluster_size='count')
    .assign(risk_score=lambda df: df['n_vulnerable'] / df['cluster_size'])
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== All clusters ranked by risk score ===')
print(cluster_stats.to_string(index=False))

# Top-K% of PACKAGES: add clusters in decreasing risk order until ~K% of packages are flagged
n_top_pkgs = max(1, round(len(results_df) * TOP_K_PCT))
top_clusters = set()
n_flagged = 0
for _, row in cluster_stats.iterrows():
    top_clusters.add(int(row['cluster']))
    n_flagged += int(row['cluster_size'])
    if n_flagged >= n_top_pkgs:
        break
print(f'\nTop clusters covering ~{TOP_K_PCT*100:.0f}% of packages ({n_flagged} / {len(results_df)}): {top_clusters}')

total_vulnerable = (results_df['label'] == 1).sum()
in_top_mask      = results_df['cluster'].isin(top_clusters)
true_positives   = ((results_df['label'] == 1) & in_top_mask).sum()
top_cluster_size = in_top_mask.sum()

recall    = true_positives / total_vulnerable if total_vulnerable else 0.0
precision = true_positives / top_cluster_size if top_cluster_size  else 0.0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print(f'\n--- Evaluation (top ~{TOP_K_PCT*100:.0f}% packages as "risky") ---')
print(f'Total packages     : {len(results_df)}')
print(f'Total vulnerable   : {total_vulnerable}')
print(f'Flagged packages   : {top_cluster_size}')
print(f'True positives     : {true_positives}')
print(f'Recall             : {recall:.4f}  ({recall*100:.1f}%)')
print(f'Precision          : {precision:.4f}  ({precision*100:.1f}%)')
print(f'F1                 : {f1:.4f}')

=== All clusters ranked by risk score ===
 cluster  n_vulnerable  cluster_size  risk_score
       3           250           260    0.961538
       1           487          1053    0.462488
       2             3             7    0.428571
       0           249           614    0.405537

Top clusters covering ~33% of packages (1313 / 1934): {1, 3}

--- Evaluation (top ~33% packages as "risky") ---
Total packages     : 1934
Total vulnerable   : 989
Flagged packages   : 1313
True positives     : 737
Recall             : 0.7452  (74.5%)
Precision          : 0.5613  (56.1%)
F1                 : 0.6403


## Step 10: Save the GNN model

In [13]:
# Save encoder weights (projection head is discarded as per the workflow)
torch.save(model.encoder.state_dict(), MODEL_OUTPUT)
joblib.dump(final_km, KMEANS_OUTPUT)
joblib.dump(scaler,   SCALER_OUTPUT)

# Save cluster risk-score table for downstream use
cluster_stats.to_csv('../models/lib/gnn_cluster_risk_scores2.csv', index=False)

print(f'Encoder saved       → {MODEL_OUTPUT}')
print(f'KMeans saved        → {KMEANS_OUTPUT}')
print(f'Scaler saved        → {SCALER_OUTPUT}')
print(f'Risk scores saved   → ../models/lib/gnn_cluster_risk_scores2.csv')

Encoder saved       → ../models/lib/gnn_encoder2.pt
KMeans saved        → ../models/lib/gnn_kmeans2.pkl
Scaler saved        → ../models/lib/gnn_scaler2.pkl
Risk scores saved   → ../models/lib/gnn_cluster_risk_scores2.csv
